# Experiment template (KAN-81)

Copy to `notebooks/<ticket>-<name>.ipynb`, fill the parameters cell, run top to bottom.
Protocol: `docs/experiments/protocol.md` (sections 9 to 12). Metrics come from code, never typed by hand.
Run: `uv run --group experiments jupyter nbconvert --to notebook --execute --inplace <file>`

In [ ]:
# Parameters (papermill-style): edit only this cell.
TICKET = "KAN-0"
AGENT = "claude-data-scientist"
RUN_NAME = "template-run"
PREREG_ID = "e00-template"  # id of the committed prereg file
SNAPSHOT_DIR = "~/pm25-experiments/snapshots/e02-v1"
SPLIT = "dev"  # dev | holdout

In [ ]:
from pathlib import Path

from pm25_forecast.experiments import tracking
from pm25_forecast.experiments.dataset import load_snapshot
from pm25_forecast.experiments.prereg import load_prereg

prereg_path = next(Path("docs/experiments/prereg").glob(f"{PREREG_ID}.*"), None)
prereg = load_prereg(prereg_path) if prereg_path else None
obs, vintages, manifest = load_snapshot(Path(SNAPSHOT_DIR).expanduser())
snapshot_sha256 = manifest["snapshot_sha256"]
git_sha, git_dirty = tracking.git_state()
print(tracking.tracking_uri(), snapshot_sha256, git_sha, git_dirty)

In [ ]:
# Fit and score here with the shared feature module and fold code; fill `metrics` from results.
with tracking.tracked_run(
    RUN_NAME,
    tags={
        "ticket": TICKET,
        "agent": AGENT,
        "model_name": prereg.model if prereg else "unset",
        "model_family": prereg.model_family if prereg else "unset",
        "feature_set": prereg.feature_set if prereg else "unset",
        "git_sha": git_sha,
        "git_dirty": git_dirty,
        "snapshot_sha256": snapshot_sha256,
        "split": SPLIT,
        "prereg": PREREG_ID,
    },
    params={"seed": 0, "ci_level": "see protocol section 8"},
) as metrics:
    metrics.update({"mae_24h": float("nan"), "rmse_24h": float("nan"), "n_pairs_24h": 0})